## Model Improvement Techniques

This notebook improves model performance using two strategies:
class weighting and interaction features.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, recall_score,
                             precision_score, f1_score,
                             roc_curve)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [11]:
# Load data with correct 44 features
model_features = joblib.load("../models/feature_names.pkl")
df = pd.read_csv("../data/processed/diabetic_engineered.csv")

X = df[model_features]
y = df['readmitted_30']

# Same 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Apply SMOTE to training only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Data loaded!")
print(f"Features: {len(model_features)}")
print(f"Training: {len(X_train_sm):,} rows (after SMOTE)")
print(f"Validation: {len(X_val):,} rows (unbalanced)")
print(f"Class imbalance ratio: {(y_train==0).sum() / (y_train==1).sum():.1f}:1")

Data loaded!
Features: 44
Training: 126,572 rows (after SMOTE)
Validation: 15,265 rows (unbalanced)
Class imbalance ratio: 8.0:1


### Strategy 1: scale_pos_weight=8
Tells XGBoost to penalize errors on minority class 8x more heavily.

Result: Recall improved from 4.2% to 39.8% (nearly 10x improvement)

ROC-AUC: 0.6332 → 0.6559 (+0.0227)

In [12]:
## --- Strategy 1 : XGBoost with scale_pos_weight ---
# This tells XGBoost about class imbalance directly
# scale_pos_weight = negative/positive = 8.0

print("Training XGBoost with scale_pos_weight=8")

xgb_weighted = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    scale_pos_weight=8,  # handles class imbalance
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

# Train on ORIGINAL training data (no SMOTE needed)
xgb_weighted.fit(X_train, y_train)

y_val_prob_w = xgb_weighted.predict_proba(X_val)[:, 1]
y_val_pred_w = (y_val_prob_w >= 0.5).astype(int)

auc_w  = roc_auc_score(y_val, y_val_prob_w)
rec_w  = recall_score(y_val, y_val_pred_w)
prec_w = precision_score(y_val, y_val_pred_w)
f1_w   = f1_score(y_val, y_val_pred_w)

print("XGBoost with scale_pos_weight done!")
print(f"Validation Results:")
print(f"  ROC-AUC: {auc_w:.4f}")
print(f"  Recall: {rec_w:.4f}")
print(f"  Precision: {prec_w:.4f}")
print(f"  F1-Score: {f1_w:.4f}")
print(f"\nVs original: 0.6332 - {auc_w:.4f} ({'+' if auc_w>0.6332 else ''}{auc_w-0.6332:.4f})")

Training XGBoost with scale_pos_weight=8
XGBoost with scale_pos_weight done!
Validation Results:
  ROC-AUC: 0.6559
  Recall: 0.3955
  Precision: 0.1931
  F1-Score: 0.2595

Vs original: 0.6332 - 0.6559 (+0.0227)


### Strategy 2: Interaction Features (7 new)
- inpatient_x_meds: number_inpatient × num_medications
- utilization_x_diagnoses: number_inpatient × number_diagnoses
- age_x_diagnoses: age × number_diagnoses
- emergency_x_inpatient: number_emergency × number_inpatient
- meds_x_stay: num_medications × time_in_hospital
- obesity_x_diabetes: OBESITY × DIABETES (SDOH interaction)
- composite_risk: sum of 5 binary risk flags

In [13]:
# Add interaction features to engineered dataset
df_improved = pd.read_csv("../data/processed/diabetic_engineered.csv")

print("Adding interaction features")

# Inpatient x Medications (high utilization + high meds = very high risk)
df_improved['inpatient_x_meds'] = (
    df_improved['number_inpatient'] * df_improved['num_medications']
)

# Prior utilization x Diagnosis burden
df_improved['utilization_x_diagnoses'] = (
    df_improved['number_inpatient'] * df_improved['number_diagnoses']
)

# Age x Diagnosis burden
df_improved['age_x_diagnoses'] = (
    df_improved['age'] * df_improved['number_diagnoses']
)

# Emergency x Inpatient (repeat emergency + inpatient = very high risk)
df_improved['emergency_x_inpatient'] = (
    df_improved['number_emergency'] * df_improved['number_inpatient']
)

# Medication complexity x Length of stay
df_improved['meds_x_stay'] = (
    df_improved['num_medications'] * df_improved['time_in_hospital']
)

# SDOH interaction — obesity x diabetes prevalence
df_improved['obesity_x_diabetes'] = (
    df_improved['OBESITY'] * df_improved['DIABETES']
)

# High risk flag — combination of multiple risk factors
df_improved['composite_risk'] = (
    df_improved['high_prior_inpatient'] +
    df_improved['high_diagnosis_burden'] +
    df_improved['high_med_burden'] +
    df_improved['long_stay'] +
    df_improved['emergency_admission']
)

print(f"Interaction features added!")
print(f"New features: 7")
print(f"Dataset shape: {df_improved.shape}")

Adding interaction features
Interaction features added!
New features: 7
Dataset shape: (101766, 52)


In [14]:
# Get all features including new interaction features
new_features = model_features + [
    'inpatient_x_meds', 'utilization_x_diagnoses',
    'age_x_diagnoses', 'emergency_x_inpatient',
    'meds_x_stay', 'obesity_x_diabetes', 'composite_risk'
]

# Make sure all features exist
new_features = [f for f in new_features if f in df_improved.columns]

X_new = df_improved[new_features]
y_new = df_improved['readmitted_30']

print(f"Total features now: {len(new_features)}")

# Same 70/15/15 split
X_train_n, X_temp_n, y_train_n, y_temp_n = train_test_split(
    X_new, y_new, test_size=0.30, random_state=42, stratify=y_new
)
X_val_n, X_test_n, y_val_n, y_test_n = train_test_split(
    X_temp_n, y_temp_n, test_size=0.50, random_state=42, stratify=y_temp_n
)

# Apply SMOTE to training only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm_n, y_train_sm_n = smote.fit_resample(X_train_n, y_train_n)

# Train XGBoost with best settings
print("\nTraining XGBoost with interaction features")

xgb_improved = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    scale_pos_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

xgb_improved.fit(X_train_n, y_train_n)

y_val_prob_imp = xgb_improved.predict_proba(X_val_n)[:, 1]
y_val_pred_imp = (y_val_prob_imp >= 0.5).astype(int)

auc_imp  = roc_auc_score(y_val_n, y_val_prob_imp)
rec_imp  = recall_score(y_val_n, y_val_pred_imp)
prec_imp = precision_score(y_val_n, y_val_pred_imp)
f1_imp   = f1_score(y_val_n, y_val_pred_imp)

print("Model with interaction features done!")
print(f"Validation Results:")
print(f"  ROC-AUC: {auc_imp:.4f}")
print(f"  Recall: {rec_imp:.4f}")
print(f"  Precision: {prec_imp:.4f}")
print(f"  F1-Score: {f1_imp:.4f}")
print(f"\nImprovement: 0.6559 - {auc_imp:.4f} ({'+' if auc_imp>0.6559 else ''}{auc_imp-0.6559:.4f})")

Total features now: 51

Training XGBoost with interaction features
Model with interaction features done!
Validation Results:
  ROC-AUC: 0.6616
  Recall: 0.3979
  Precision: 0.1990
  F1-Score: 0.2653

Improvement: 0.6559 - 0.6616 (+0.0057)


### Improvement Journey
- Original XGBoost: 0.6332
- After tuning: 0.6362 (+0.0030)
- After scale_pos_weight: 0.6559 (+0.0227)
- After interaction features: 0.6616 (+0.0284 total)

In [18]:
# Final comparison of all approaches
comparison = pd.DataFrame({
    'Model': [
        'XGBoost original (SMOTE)',
        'XGBoost scale_pos_weight',
        'XGBoost interaction features'
    ],
    'ROC-AUC': [0.6332, 0.6559, 0.6616],
    'Recall':  [0.0423, 0.3955, 0.3979],
    'Precision':[0.3172, 0.1931, 0.1990],
    'F1':      [0.0746, 0.2595, 0.2653]
})

print("-" * 75)
print("FINAL MODEL COMPARISON")
print("-" * 75)
print(comparison.to_string(index=False))
print("-" * 75)
print(f"Best ROC-AUC: XGBoost interaction features — 0.6616")

comparison.to_csv('../dashboard/improvement_comparison.csv', index=False)
print("Comparison saved!")

---------------------------------------------------------------------------
FINAL MODEL COMPARISON
---------------------------------------------------------------------------
                       Model  ROC-AUC  Recall  Precision     F1
    XGBoost original (SMOTE)   0.6332  0.0423     0.3172 0.0746
    XGBoost scale_pos_weight   0.6559  0.3955     0.1931 0.2595
XGBoost interaction features   0.6616  0.3979     0.1990 0.2653
---------------------------------------------------------------------------
Best ROC-AUC: XGBoost interaction features — 0.6616
Comparison saved!
